# SignalForge Week 11 ORPO Colab Notebook

This notebook is the explicit Colab training path for the current SignalForge repo.

It assumes the repo is available at `/content/SignalForge` and uses the new export bundle created by `training/export_unsloth_datasets.py`.

Training flow:

1. Install Unsloth + TRL dependencies.
2. Export the repo's train/dev/held-out preference bundle.
3. Load `preferences_train.jsonl` and `preferences_dev.jsonl`.
4. Optionally run a short SFT warm start on `sft_text`.
5. Run ORPO on `prompt` / `chosen` / `rejected`.
6. Save the LoRA adapter.
7. Keep `held_out` sealed until the recipe is locked.


In [ ]:
# Run this on a fresh Colab runtime.
# Do not install trl / transformers / peft separately afterwards.
!pip install --upgrade pip setuptools wheel
!pip install --no-cache-dir --force-reinstall "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


In [ ]:
# Restart immediately after install so Colab loads the new package set cleanly.
import os
os.kill(os.getpid(), 9)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Repo Setup

This cell can clone the repo automatically if `/content/SignalForge` does not exist yet.


In [ ]:
from pathlib import Path
import json
import os
import subprocess

REPO_URL = 'https://github.com/nebiyuephrata/SignalForge.git'
REPO_BRANCH = 'main'
REPO_ROOT = Path('/content/SignalForge')

if not REPO_ROOT.exists():
    subprocess.run([
        'git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_ROOT)
    ], check=True)
else:
    print(f'Reusing existing repo at {REPO_ROOT}')

TRAIN_EXPORT = REPO_ROOT / 'training_data' / 'unsloth' / 'preferences_train.jsonl'
DEV_EXPORT = REPO_ROOT / 'training_data' / 'unsloth' / 'preferences_dev.jsonl'
HELD_OUT_EXPORT = REPO_ROOT / 'training_data' / 'unsloth' / 'preferences_held_out.jsonl'
OUTPUT_DIR = REPO_ROOT / 'training' / 'colab_orpo_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({'repo_root': str(REPO_ROOT), 'output_dir': str(OUTPUT_DIR)})


In [ ]:
%cd /content/SignalForge
!python training/export_unsloth_datasets.py


In [ ]:
from datasets import Dataset

def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl(TRAIN_EXPORT)
dev_rows = read_jsonl(DEV_EXPORT)

print({'train_rows': len(train_rows), 'dev_rows': len(dev_rows)})
print(train_rows[0]['task_id'], train_rows[0]['dimension'])
print(train_rows[0]['prompt'][:300])


## Optional Warm Start

Leave `RUN_SFT_WARMSTART = False` unless the base model is too generic or fails to keep the email shape stable.

This warm start should stay short. It is not the main Path B optimization.


In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1536
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
RUN_SFT_WARMSTART = False

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

print(type(model))
print({'pad_token': tokenizer.pad_token, 'eos_token': tokenizer.eos_token})


In [ ]:
if RUN_SFT_WARMSTART:
    from trl import SFTTrainer, SFTConfig

    warm_train = Dataset.from_list([{'text': row['sft_text']} for row in train_rows])
    warm_dev = Dataset.from_list([{'text': row['sft_text']} for row in dev_rows])

    sft_trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=warm_train,
        eval_dataset=warm_dev,
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        args=SFTConfig(
            output_dir=str(OUTPUT_DIR / 'sft_warmstart'),
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            max_steps=80,
            learning_rate=2e-4,
            logging_steps=5,
            eval_strategy='steps',
            eval_steps=20,
            save_steps=40,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            report_to='none',
        ),
    )
    sft_trainer.train()
else:
    print('Skipping SFT warm start.')


## ORPO Training

This is the main Path B step.

The exported rows already contain plain-string `prompt`, `chosen`, and `rejected` fields, which is the shape ORPO expects.


In [ ]:
from trl import ORPOConfig, ORPOTrainer

orpo_train = Dataset.from_list([
    {
        'prompt': row['prompt'],
        'chosen': row['chosen'],
        'rejected': row['rejected'],
        'task_id': row['task_id'],
        'dimension': row['dimension'],
    }
    for row in train_rows
])

orpo_dev = Dataset.from_list([
    {
        'prompt': row['prompt'],
        'chosen': row['chosen'],
        'rejected': row['rejected'],
        'task_id': row['task_id'],
        'dimension': row['dimension'],
    }
    for row in dev_rows
])

orpo_args = ORPOConfig(
    output_dir=str(OUTPUT_DIR / 'orpo_run'),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    num_train_epochs=2,
    logging_steps=5,
    save_steps=50,
    eval_strategy='steps',
    eval_steps=25,
    max_length=1536,
    max_prompt_length=1024,
    beta=0.1,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to='none',
    remove_unused_columns=False,
)

trainer = ORPOTrainer(
    model=model,
    args=orpo_args,
    train_dataset=orpo_train,
    eval_dataset=orpo_dev,
    tokenizer=tokenizer,
)

trainer.train()


In [ ]:
adapter_dir = OUTPUT_DIR / 'orpo_adapter'
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print({'adapter_dir': str(adapter_dir)})


In [ ]:
FastLanguageModel.for_inference(model)
sample_prompt = train_rows[0]['prompt']
inputs = tokenizer([sample_prompt], return_tensors='pt', truncation=True).to('cuda')
outputs = model.generate(**inputs, max_new_tokens=180, temperature=0.7, do_sample=True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=False)[0])


## Evaluation Discipline

- Tune only on `train` and `dev`.
- Do not inspect `preferences_held_out.jsonl` until the recipe is frozen.
- After the best run, generate held-out outputs once and score them back in the repo.

Recommended next loop:

1. Run one ORPO smoke test.
2. Inspect dev behavior on `weak_confidence`, `signal_over_claiming`, `pricing_handoff`, and `cto_sensitivity`.
3. Add harder negatives.
4. Rerun ORPO or switch to SimPO.
